# Prediciendo diabetes con Boosting



## Paso 1: Carga del conjunto de datos



In [1]:
import pandas as pd
import numpy as np


url = "https://breathecode.herokuapp.com/asset/internal-link?id=930&path=diabetes.csv"
df = pd.read_csv(url)


cols_with_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in cols_with_zero:
    df[col] = df[col].replace(0, np.nan)
df.fillna(df.median(), inplace=True)


from sklearn.model_selection import train_test_split

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Datos cargados y divididos:")
print(f"  - Entrenamiento: {X_train.shape[0]} muestras")
print(f"  - Prueba: {X_test.shape[0]} muestras")

Datos cargados y divididos:
  - Entrenamiento: 614 muestras
  - Prueba: 154 muestras


## Paso 2: Construye un Boosting



In [ ]:
# Instalar xgboost en el entorno del kernel (ejecutar si da ModuleNotFoundError, luego reiniciar kernel)
%pip install xgboost -q

In [2]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt


xgb_base = XGBClassifier(random_state=42)
xgb_base.fit(X_train, y_train)
y_pred_base = xgb_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)

print("=== XGBoost (configuración por defecto) ===")
print(f"Accuracy: {acc_base:.4f}")
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_base))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_base))

ModuleNotFoundError: No module named 'xgboost'

In [ ]:

n_estimators_values = [50, 100, 150, 200, 300, 500]
accuracies_n_est = []

for n in n_estimators_values:
    xgb = XGBClassifier(n_estimators=n, random_state=42)
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies_n_est.append(acc)
    print(f"n_estimators={n}: Accuracy = {acc:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(n_estimators_values, accuracies_n_est, "o-", linewidth=2, markersize=8)
plt.xlabel("n_estimators (número de rounds de boosting)")
plt.ylabel("Accuracy")
plt.title("Impacto de n_estimators en la precisión del XGBoost")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

max_depth_values = [3, 5, 7, 10]
learning_rates = [0.01, 0.05, 0.1, 0.2]
accuracies_lr = []

for lr in learning_rates:
    xgb = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=lr,
        random_state=42
    )
    xgb.fit(X_train, y_train)
    acc = accuracy_score(y_test, xgb.predict(X_test))
    accuracies_lr.append(acc)
    print(f"learning_rate={lr}: Accuracy = {acc:.4f}")

plt.figure(figsize=(8, 5))
plt.bar([str(lr) for lr in learning_rates], accuracies_lr, color="coral", edgecolor="black")
plt.xlabel("learning_rate")
plt.ylabel("Accuracy")
plt.title("Impacto de learning_rate en la precisión del XGBoost")
plt.tight_layout()
plt.show()


accuracies_depth = []
for depth in max_depth_values:
    xgb = XGBClassifier(
        n_estimators=100, max_depth=depth,
        random_state=42
    )
    xgb.fit(X_train, y_train)
    acc = accuracy_score(y_test, xgb.predict(X_test))
    accuracies_depth.append(acc)
    print(f"max_depth={depth}: Accuracy = {acc:.4f}")

plt.figure(figsize=(8, 5))
plt.bar([str(d) for d in max_depth_values], accuracies_depth, color="seagreen", edgecolor="black")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Impacto de max_depth en la precisión del XGBoost")
plt.tight_layout()
plt.show()

In [ ]:

import seaborn as sns

n_est_range = [100, 200, 300]
depth_range = [3, 5, 7, 10]
results = []

for n in n_est_range:
    row = []
    for d in depth_range:
        xgb = XGBClassifier(
            n_estimators=n, max_depth=d,
            random_state=42
        )
        xgb.fit(X_train, y_train)
        acc = accuracy_score(y_test, xgb.predict(X_test))
        row.append(acc)
    results.append(row)

plt.figure(figsize=(8, 6))
sns.heatmap(
    results,
    xticklabels=depth_range,
    yticklabels=n_est_range,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    cbar_kws={"label": "Accuracy"},
)
plt.xlabel("max_depth")
plt.ylabel("n_estimators")
plt.title("Accuracy según n_estimators y max_depth (XGBoost)")
plt.tight_layout()
plt.show()


best_xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=42
)
best_xgb.fit(X_train, y_train)
y_pred_xgb = best_xgb.predict(X_test)
print(f"\n=== Modelo XGBoost optimizado ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")

## Paso 3: Guarda el modelo



In [ ]:
import joblib
from pathlib import Path


cwd = Path(".").resolve()
project_root = cwd if (cwd / "src").exists() else cwd.parent
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True, parents=True)


model_path = models_dir / "xgboost_diabetes.pkl"
joblib.dump(best_xgb, model_path)

print(f"Modelo guardado en: {model_path}")

## Paso 4: Análisis y comparación de los tres modelos



In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid_dt = {
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy", "log_loss"],
}
grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=5,
    scoring="accuracy",
)
grid_dt.fit(X_train, y_train)
modelo_dt = grid_dt.best_estimator_


modelo_rf = joblib.load(models_dir / "random_forest_diabetes.pkl")


modelo_xgb = best_xgb


y_pred_dt = modelo_dt.predict(X_test)
y_pred_rf = modelo_rf.predict(X_test)
y_pred_xgb = modelo_xgb.predict(X_test)

print("Modelos cargados/entrenados correctamente.")

In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score

modelos_nombres = ["Árbol de decisión", "Random Forest", "XGBoost"]
predicciones = [y_pred_dt, y_pred_rf, y_pred_xgb]

resultados = []
for nombre, y_pred in zip(modelos_nombres, predicciones):
    acc = accuracy_score(y_test, y_pred)
    prec_0 = precision_score(y_test, y_pred, pos_label=0, zero_division=0)
    prec_1 = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec_0 = recall_score(y_test, y_pred, pos_label=0, zero_division=0)
    rec_1 = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    resultados.append({
        "Modelo": nombre,
        "Accuracy": acc,
        "Precision_Clase0 (No diabetes)": prec_0,
        "Precision_Clase1 (Diabetes)": prec_1,
        "Recall_Clase0": rec_0,
        "Recall_Clase1": rec_1,
    })

df_resultados = pd.DataFrame(resultados)
print("=== Comparación de los tres modelos ===\n")
print(df_resultados.to_string(index=False))

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 10))


axes[0, 0].bar(modelos_nombres, df_resultados["Accuracy"], color=["#3498db", "#2ecc71", "#e74c3c"])
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].set_title("Accuracy por modelo")
axes[0, 0].tick_params(axis="x", rotation=15)


x = np.arange(len(modelos_nombres))
width = 0.35
axes[0, 1].bar(x - width/2, df_resultados["Precision_Clase0 (No diabetes)"], width, label="Clase 0 (No diabetes)")
axes[0, 1].bar(x + width/2, df_resultados["Precision_Clase1 (Diabetes)"], width, label="Clase 1 (Diabetes)")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(modelos_nombres, rotation=15)
axes[0, 1].set_ylabel("Precisión")
axes[0, 1].set_title("Precisión por clase en cada modelo")
axes[0, 1].legend()

axes[0, 2].axis("off")


for i, (nombre, y_pred) in enumerate(zip(modelos_nombres, predicciones)):
    ax = axes[1, i]
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"{nombre}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_xticklabels(["No diabetes", "Diabetes"])
    ax.set_yticklabels(["No diabetes", "Diabetes"])

plt.tight_layout()
plt.show()

In [ ]:

print("=== Análisis por clase ===\n")

for i, nombre in enumerate(modelos_nombres):
    prec_0 = df_resultados.loc[i, "Precision_Clase0 (No diabetes)"]
    prec_1 = df_resultados.loc[i, "Precision_Clase1 (Diabetes)"]
    
    if prec_0 >= prec_1:
        mejor_clase = "Clase 0 (No diabetes)"
        peor_clase = "Clase 1 (Diabetes)"
        mejor_prec = prec_0
        peor_prec = prec_1
    else:
        mejor_clase = "Clase 1 (Diabetes)"
        peor_clase = "Clase 0 (No diabetes)"
        mejor_prec = prec_1
        peor_prec = prec_0
    
    print(f"{nombre}:")
    print(f"  - Mayor precisión: {mejor_clase} ({mejor_prec:.4f})")
    print(f"  - Menor precisión: {peor_clase} ({peor_prec:.4f})")
    print()

In [ ]:

mejor_idx = df_resultados["Accuracy"].idxmax()
mejor_modelo = df_resultados.loc[mejor_idx, "Modelo"]
mejor_accuracy = df_resultados.loc[mejor_idx, "Accuracy"]

print("=== CONCLUSIÓN ===\n")
print(f"El modelo con mejor accuracy global es: {mejor_modelo} ({mejor_accuracy:.4f})")
print()
print("Recomendación: Dado el análisis comparativo de los tres enfoques (Árbol de decisión,")
print("Random Forest y Boosting/XGBoost), el modelo elegido es el que mejor equilibra")
print("accuracy, precisión por clase y capacidad de generalización.")
print()
print(f"En este caso: **{mejor_modelo}** es el modelo recomendado para la predicción de diabetes.")